# Demo 8 - Sign-in volume forecast with anomaly band

**Fast** (daily counts, a few dozen points) · **Pool:** Small · **Visual:** actual vs forecast

**The question:** is today's sign-in volume actually unusual, or does it just feel high?

We count sign-ins per day, fit a model that learns the underlying level, the direction of
travel and the weekly rhythm, then project the next week and shade the range that counts as
normal variation. Days that already fell outside that range are marked in red.

KQL has one built-in forecast function and you take what it gives you. Here you choose the
model, tune the seasonality, inspect how wrong it was, and plot the uncertainty.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `LOOKBACK_DAYS` - 60 days gives roughly eight weekly cycles, which is about the minimum
  for the model to learn a weekly rhythm with any confidence.
- `FORECAST_DAYS` - how far ahead to project.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 60     # enough history for weekly seasonality; still only ~60 rows
FORECAST_DAYS = 7

## 3. Reduce the whole workspace to one number per day

Count sign-ins per calendar day. That is it - about 60 rows.

`asfreq("D")` then fills in any day that had no sign-ins at all with a zero. Without it, a
completely quiet day would silently vanish from the series and shift every later point
along by one, which would quietly wreck the seasonality.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

df = data_provider.read_table("SigninLogs", WORKSPACE)
daily = (df.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
           .groupBy(F.to_date("TimeGenerated").alias("day")).agg(F.count("*").alias("signins"))
           .orderBy("day")).toPandas()

if daily.empty:
    s = pd.Series(dtype="float64")
else:
    s = daily.set_index(pd.to_datetime(daily["day"]))["signins"].asfreq("D").fillna(0)
print("days:", len(s))

## 4. Forecast next week, and mark what was already abnormal

**Holt-Winters exponential smoothing** learns three things from the history at once:

- the **level** - roughly how many sign-ins we see on a normal day
- the **trend** - whether that level is drifting up or down over time
- the **seasonality** - the repeating weekly shape, busy Monday to Friday and quiet at
  the weekend

It then projects all three forward together. `seasonal_periods=7` is what tells it the
rhythm repeats weekly rather than daily or monthly.

The shaded band is roughly a 95% confidence interval, built from how far the model's own
fitted values missed the actual history. Read it as "normal variation": a point inside the
band is unremarkable no matter how spiky it looks to the eye.

Red dots mark days in the **past** that fell outside the band. Those are the days worth
asking about, and they are also the honest test of whether the model has learned your
organisation or is just drawing a smooth line through the middle.

The cell skips with a message if there is not enough history for two full weekly cycles.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

SEASON = 7
if len(s) < 2 * SEASON + 1:
    print(f"Only {len(s)} daily points - Holt-Winters needs at least {2*SEASON+1} for a "
          "weekly season. Raise LOOKBACK_DAYS and re-run.")
else:
    model = ExponentialSmoothing(s, trend="add", seasonal="add", seasonal_periods=SEASON).fit()
    fc = model.forecast(FORECAST_DAYS)
    resid_std = float(np.std(s - model.fittedvalues))
    future = pd.date_range(s.index[-1] + pd.Timedelta(days=1), periods=FORECAST_DAYS)

    plt.figure(figsize=(13,5))
    plt.plot(s.index, s.values, label="actual", color="#2c3e50")
    plt.plot(future, fc.values, label="forecast", color="#2980b9", marker="o")
    plt.fill_between(future, fc.values-1.96*resid_std, fc.values+1.96*resid_std,
                     color="#2980b9", alpha=.2, label="95% band")
    anom = s[np.abs(s - model.fittedvalues) > 1.96*resid_std]
    plt.scatter(anom.index, anom.values, color="red", zorder=5, label="historical anomalies")
    plt.title("Daily sign-in volume: actual, forecast and anomaly band")
    plt.ylabel("sign-ins/day"); plt.legend(); plt.tight_layout(); plt.show()

## Why a notebook beats KQL here

Holt-Winters exponential smoothing with additive trend and 7-day seasonality is a statistical model, not a query. You can tune it, extract residuals, and plot a band - control KQL's single built-in forecast verb doesn't give you.